# 03 — PDF manifest, AyurGenix check, and **PDF text extraction**

**What this notebook does**
1. **`rag_pdf_manifest`** — registers each Ayurveda reference PDF on the Volume.
2. **Explore `ayurgenix_curated`** — quick sanity check of the **primary** Kaggle-backed table (notebook 01–02).
3. **`pdf_text_chunks`** — extracts **searchable text** from all three PDFs (page-aware, chunked) into Delta as the **secondary** knowledge layer for your chatbot.

**PDFs on the Volume (after notebook 00)**  
`ayurveda.pdf`, `ayurveda_offering_herbal_healing.pdf`, `ayurveda_the_science_of_life_dossier.pdf`

**How to run** — Serverless → **Run all**. `%pip` installs **`pypdf`**; if `%pip` is blocked, ask an admin to add **pypdf** to the cluster image.


In [0]:
%sql
USE CATALOG ayurveda_lakehouse;
USE SCHEMA ayurgenix;


## Part A — Manifest of PDF objects for RAG
Paths match the three Ayurveda PDFs under `raw_data/` (copied to the Volume in notebook 00).


In [0]:
%sql
CREATE OR REPLACE TABLE ayurveda_lakehouse.ayurgenix.rag_pdf_manifest AS
SELECT * FROM VALUES
  ('ayurveda.pdf', '/Volumes/ayurveda_lakehouse/ayurgenix/source_vault/ayurveda.pdf'),
  ('ayurveda_offering_herbal_healing.pdf', '/Volumes/ayurveda_lakehouse/ayurgenix/source_vault/ayurveda_offering_herbal_healing.pdf'),
  ('ayurveda_the_science_of_life_dossier.pdf', '/Volumes/ayurveda_lakehouse/ayurgenix/source_vault/ayurveda_the_science_of_life_dossier.pdf')
AS t(file_name, volume_path);


## Part B — Explore structured AyurGenix data (**primary** chatbot source)
Rows come from **`AyurGenixAI_Dataset.csv`** (notebooks 01–02).


In [0]:
%sql
SELECT * FROM ayurveda_lakehouse.ayurgenix.ayurgenix_curated LIMIT 20;


## Part C — Extract useful text from all PDFs (**secondary** RAG source)
Each PDF is read with **`binaryFile`**, parsed with **`pypdf`**, split into **`CHUNK_CHARS`** segments (page + chunk index preserved), and written to **`pdf_text_chunks`**. Use CSV-first retrieval, then enrich from **`chunk_text`**.


In [0]:
%pip install -q pypdf


**Note:** No OCR — image-only scans may return little text. For those, add OCR in a later pipeline.


In [0]:
import io
from pypdf import PdfReader

VOLUME_ROOT = "/Volumes/ayurveda_lakehouse/ayurgenix/source_vault"
PDF_FILES = [
    "ayurveda.pdf",
    "ayurveda_offering_herbal_healing.pdf",
    "ayurveda_the_science_of_life_dossier.pdf",
]
CHUNK_CHARS = 2000
FQN_CHUNKS = "ayurveda_lakehouse.ayurgenix.pdf_text_chunks"

def rows_for_pdf(spark, file_name: str):
    full_uri = f"{VOLUME_ROOT}/{file_name}"
    bin_df = spark.read.format("binaryFile").load(full_uri)
    row = bin_df.select("content").limit(1).collect()
    if not row:
        print(f"WARN: no binary rows for {file_name}")
        return []
    raw = row[0]["content"]
    reader = PdfReader(io.BytesIO(raw))
    out = []
    for page_idx, page in enumerate(reader.pages):
        text = (page.extract_text() or "").strip()
        if not text:
            out.append({
                "source_file": file_name,
                "page_number": page_idx + 1,
                "chunk_index": 0,
                "chunk_text": "",
                "char_count": 0,
            })
            continue
        for start in range(0, len(text), CHUNK_CHARS):
            chunk = text[start : start + CHUNK_CHARS]
            out.append({
                "source_file": file_name,
                "page_number": page_idx + 1,
                "chunk_index": start // CHUNK_CHARS,
                "chunk_text": chunk,
                "char_count": len(chunk),
            })
    return out

all_rows = []
for fn in PDF_FILES:
    all_rows.extend(rows_for_pdf(spark, fn))

if not all_rows:
    raise RuntimeError("No PDF rows produced; confirm PDFs landed on the Volume in notebook 00.")

chunks_df = spark.createDataFrame(all_rows)
(
    chunks_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FQN_CHUNKS)
)

print("Wrote table", FQN_CHUNKS)
spark.sql(f"SELECT source_file, COUNT(*) AS chunks, SUM(char_count) AS total_chars FROM {FQN_CHUNKS} GROUP BY source_file ORDER BY source_file").show(truncate=False)


## Part D — Did we get **useful** text from the PDFs?

Heuristics (tunable in the Python cell below):

- **`chunks_with_useful_text`**: chunks where `length(trim(chunk_text)) >= 50` (enough characters to be a real snippet).
- **`total_chars`**: sum of `char_count` per file — very low values often mean **scanned** PDFs with no text layer (needs OCR).
- **`pct_substantive`**: share of chunks that pass the 50-character rule.

The Python cell prints **OK** vs **LOW_TEXT** / **WEAK** per file before you move on to embeddings.


In [0]:
%sql
SELECT
  source_file,
  COUNT(*) AS total_chunks,
  SUM(CASE WHEN length(trim(chunk_text)) >= 50 THEN 1 ELSE 0 END) AS chunks_with_useful_text,
  SUM(char_count) AS total_chars,
  ROUND(100.0 * SUM(CASE WHEN length(trim(chunk_text)) >= 50 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_substantive
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
GROUP BY source_file
ORDER BY source_file;


In [0]:

MIN_TOTAL_CHARS = 200
MIN_USEFUL_CHUNKS = 1

q = spark.sql('''
SELECT source_file,
  SUM(CASE WHEN length(trim(chunk_text)) >= 50 THEN 1 ELSE 0 END) AS useful_chunks,
  SUM(char_count) AS total_chars
FROM ayurveda_lakehouse.ayurgenix.pdf_text_chunks
GROUP BY source_file
ORDER BY source_file
''')

print("--- PDF extraction quality (automated check) ---")
for r in q.collect():
    name = r["source_file"]
    uc = int(r["useful_chunks"] or 0)
    tc = int(r["total_chars"] or 0)
    if uc >= MIN_USEFUL_CHUNKS and tc >= MIN_TOTAL_CHARS:
        status = "OK — enough text for typical RAG / keyword retrieval"
    elif tc < MIN_TOTAL_CHARS:
        status = "LOW_TEXT — likely scanned PDF or empty text layer; try OCR or a different source PDF"
    else:
        status = "WEAK — few substantive chunks; inspect vw_rag_pdf_chunks_sample (notebook 04)"
    print("{}: useful_chunks={}, total_chars={} => {}".format(name, uc, tc, status))


**Next step:** open `04_create_chatbot_analytics_views` — views span **curated CSV + PDF chunks**.
